In [1]:
import pandas as pd
import json
from rich import print
from rich.pretty import Pretty

from datetime import datetime
pd.set_option('display.max_columns', None)

_EXTRACTED_META_COLS = ["_filter_param", "_filter_value", "_extract_datetime"]

def _add_openalex_extracted_metadata(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in _EXTRACTED_META_COLS:
        if col not in df.columns:
            df[col] = pd.NA
    return df

def _add_openalex_loaded_metadata(df: pd.DataFrame, load_datetime=None) -> pd.DataFrame:
    df = df.copy()
    if load_datetime is None:
        load_datetime = datetime.today()
    df["_load_datetime"] = pd.to_datetime(load_datetime)
    return df


In [2]:
df_work_raw = catalog.load('raw/openalex/work_dev#parquet')

[03/04/26 11:02:45] INFO     Loading data from raw/openalex/work_dev#parquet                   ]8;id=499426;file:///home/pablo/dev/scholar/kedro-scholar/.venv/lib/python3.10/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=352409;file:///home/pablo/dev/scholar/kedro-scholar/.venv/lib/python3.10/site-packages/kedro/io/data_catalog.py#1048\1048]8;;\
                             (ParquetDataset)...                                                                   

**Estructura de Authorships**

In [3]:
from rich import print, pretty

first_authorship = df_work_raw['authorships'].iloc[0][0]
print(pretty.Pretty(first_authorship, expand_all=False))

{
    'affiliations': array([{'institution_ids': array(['https://openalex.org/I1286329397',
              'https://openalex.org/I4391768179'], dtype=object), 'raw_affiliation_string': 'U.S. Geological 
Survey, Fort Collins Science Center, Jemez Mountains Field Station, Los Alamos, NM 87544, USA'},
       {'institution_ids': array(['https://openalex.org/I4391768179'], dtype=object), 'raw_affiliation_string': 
'Fort Collins Science Center (2150 Centre Avenue,Fort Collins,CO80526-8118 - United States)'}],
      dtype=object),
    'author': {
        'display_name': 'Craig D. Allen',
        'id': 'https://openalex.org/A5070307451',
        'orcid': 'https://orcid.org/0000-0002-8777-5989'
    },
    'author_position': 'first',
    'countries': array(['US'], dtype=object),
    'institutions': array([{'country_code': 'US', 'display_name': 'United States Geological Survey', 'id': 
'https://openalex.org/I1286329397', 'lineage': array(['https://openalex.org/I1286329397',
              'https://openalex.org/I1335927249'], dtype=object), 'ror': 'https://ror.org/035a68863', 'type': 
'government'}                                                     ,
       {'country_code': None, 'display_name': 'Fort Collins Science Center', 'id': 
'https://openalex.org/I4391768179', 'lineage': array(['https://openalex.org/I1286329397',
              'https://openalex.org/I1335927249',
              'https://openalex.org/I4391768179'], dtype=object), 'ror': 'https://ror.org/00zf0nh29', 'type': 
'facility'}                                                   ],
      dtype=object),
    'is_corresponding': True,
    'raw_affiliation_strings': array(['U.S. Geological Survey, Fort Collins Science Center, Jemez Mountains Field 
Station, Los Alamos, NM 87544, USA',
       'Fort Collins Science Center (2150 Centre Avenue,Fort Collins,CO80526-8118 - United States)'],
      dtype=object),
    'raw_author_name': 'Craig D. Allen'
}

# Nodo

In [22]:
def openalex_load_work_authorships(df_work_raw):

    df_work_raw = _add_openalex_extracted_metadata(df_work_raw)

    # Seleccionar las columnas necesarias y convertir los tipos de datos
    df_work2authorships = df_work_raw[['id', 'authorships', '_filter_param', '_filter_value', '_extract_datetime']].convert_dtypes()
    df_work2authorships.rename(columns={"id": "work_id"}, inplace=True)

    # Expandir la lista de authorships
    df_work2authorships_exploded = df_work2authorships.explode('authorships', ignore_index=True)

    # Normalizar la información de authorships
    df_authorships_norm = pd.json_normalize(df_work2authorships_exploded['authorships'])
    df_authorships_norm.rename(columns={"author.id": "author_id"}, inplace=True)
    
    # Combinar work_id con la información normalizada de authorships
    df_work2authorships = df_work2authorships_exploded[['work_id', '_filter_param', '_filter_value', '_extract_datetime']].join(df_authorships_norm)

    # Extraer la relación work-author
    df_work2author = df_work2authorships[['work_id', 'author_id', 'author_position', '_filter_param', '_filter_value', '_extract_datetime']]

    # Expandir la lista de instituciones asociadas a cada autor
    df_work2institution_exploded = df_work2authorships.explode('institutions', ignore_index=True)

    # Normalizar la información de instituciones
    df_institution_norm = pd.json_normalize(df_work2institution_exploded['institutions'])
    df_institution_norm.drop(columns=['lineage'], errors='ignore', inplace=True)

    # Combinar author_id con la información normalizada de instituciones
    df_author2institution = df_work2institution_exploded[['author_id', '_filter_param', '_filter_value', '_extract_datetime']].join(df_institution_norm)

    # Combinar work_id con la información normalizada de instituciones
    df_work2institution = df_work2institution_exploded[['work_id', '_filter_param', '_filter_value', '_extract_datetime']].join(df_institution_norm)
    
    df_work2author = _add_openalex_loaded_metadata(df_work2author)
    df_work2institution = _add_openalex_loaded_metadata(df_work2institution)
    df_author2institution = _add_openalex_loaded_metadata(df_author2institution)

    return df_work2author, df_work2institution, df_author2institution


## Ejecuto Nodo

In [23]:
df_work2author, df_work2institution, df_author2institution = openalex_load_work_authorships(df_work_raw)

# Resultados

In [24]:
df_work2author

,work_id,author_id,author_position,_filter_param,_filter_value,_extract_datetime,_load_datetime
0,https://openalex.org/W2140131090,https://openalex.org/A5070307451,first,institutions.ror,https://ror.org/03cqe8w59,2026-03-03,2026-03-04 11:02:48.133527
1,https://openalex.org/W2140131090,https://openalex.org/A5042301595,middle,institutions.ror,https://ror.org/03cqe8w59,2026-03-03,2026-03-04 11:02:48.133527
2,https://openalex.org/W2140131090,https://openalex.org/A5019583393,middle,institutions.ror,https://ror.org/03cqe8w59,2026-03-03,2026-03-04 11:02:48.133527
3,https://openalex.org/W2140131090,https://openalex.org/A5076732012,middle,institutions.ror,https://ror.org/03cqe8w59,2026-03-03,2026-03-04 11:02:48.133527
4,https://openalex.org/W2140131090,https://openalex.org/A5007050438,middle,institutions.ror,https://ror.org/03cqe8w59,2026-03-03,2026-03-04 11:02:48.133527
...,...,...,...,...,...,...,...
5162,https://openalex.org/W3036911563,https://openalex.org/A5062334330,middle,institutions.ror,https://ror.org/03cqe8w59,2026-03-03,2026-03-04 11:02:48.133527
5163,https://openalex.org/W3036911563,https://openalex.org/A5062318046,last,institutions.ror,https://ror.org/03cqe8w59,2026-03-03,2026-03-04 11:02:48.133527
5164,https://openalex.org/W3095652295,https://openalex.org/A5049448648,first,institutions.ror,https://ror.org/03cqe8w59,2026-03-03,2026-03-04 11:02:48.133527
5165,https://openalex.org/W3095652295,https://openalex.org/A5063261232,middle,institutions.ror,https://ror.org/03cqe8w59,2026-03-03,2026-03-04 11:02:48.133527


In [25]:
df_work2institution

,work_id,_filter_param,_filter_value,_extract_datetime,country_code,display_name,id,ror,type,_load_datetime
0,https://openalex.org/W2140131090,institutions.ror,https://ror.org/03cqe8w59,2026-03-03,US,United States Geological Survey,https://openalex.org/I1286329397,https://ror.org/035a68863,government,2026-03-04 11:02:48.136572
1,https://openalex.org/W2140131090,institutions.ror,https://ror.org/03cqe8w59,2026-03-03,None,Fort Collins Science Center,https://openalex.org/I4391768179,https://ror.org/00zf0nh29,facility,2026-03-04 11:02:48.136572
2,https://openalex.org/W2140131090,institutions.ror,https://ror.org/03cqe8w59,2026-03-03,US,University of Arizona,https://openalex.org/I138006243,https://ror.org/03m2x1q45,education,2026-03-04 11:02:48.136572
3,https://openalex.org/W2140131090,institutions.ror,https://ror.org/03cqe8w59,2026-03-03,DZ,University of Batna 1,https://openalex.org/I162489102,https://ror.org/04hrbe508,education,2026-03-04 11:02:48.136572
4,https://openalex.org/W2140131090,institutions.ror,https://ror.org/03cqe8w59,2026-03-03,US,Oregon State University,https://openalex.org/I131249849,https://ror.org/00ysfqy60,education,2026-03-04 11:02:48.136572
...,...,...,...,...,...,...,...,...,...,...
7705,https://openalex.org/W3095652295,institutions.ror,https://ror.org/03cqe8w59,2026-03-03,AR,Universidad Nacional de La Plata,https://openalex.org/I874386039,https://ror.org/01tjs6929,education,2026-03-04 11:02:48.136572
7706,https://openalex.org/W3095652295,institutions.ror,https://ror.org/03cqe8w59,2026-03-03,AR,Consejo Nacional de Investigaciones Científica...,https://openalex.org/I151201029,https://ror.org/03cqe8w59,funder,2026-03-04 11:02:48.136572
7707,https://openalex.org/W3095652295,institutions.ror,https://ror.org/03cqe8w59,2026-03-03,AR,Universidad Nacional de La Plata,https://openalex.org/I874386039,https://ror.org/01tjs6929,education,2026-03-04 11:02:48.136572
7708,https://openalex.org/W3095652295,institutions.ror,https://ror.org/03cqe8w59,2026-03-03,AR,Universidad Nacional de La Plata,https://openalex.org/I874386039,https://ror.org/01tjs6929,education,2026-03-04 11:02:48.136572


In [26]:
df_author2institution

,author_id,_filter_param,_filter_value,_extract_datetime,country_code,display_name,id,ror,type,_load_datetime
0,https://openalex.org/A5070307451,institutions.ror,https://ror.org/03cqe8w59,2026-03-03,US,United States Geological Survey,https://openalex.org/I1286329397,https://ror.org/035a68863,government,2026-03-04 11:02:48.140046
1,https://openalex.org/A5070307451,institutions.ror,https://ror.org/03cqe8w59,2026-03-03,None,Fort Collins Science Center,https://openalex.org/I4391768179,https://ror.org/00zf0nh29,facility,2026-03-04 11:02:48.140046
2,https://openalex.org/A5042301595,institutions.ror,https://ror.org/03cqe8w59,2026-03-03,US,University of Arizona,https://openalex.org/I138006243,https://ror.org/03m2x1q45,education,2026-03-04 11:02:48.140046
3,https://openalex.org/A5019583393,institutions.ror,https://ror.org/03cqe8w59,2026-03-03,DZ,University of Batna 1,https://openalex.org/I162489102,https://ror.org/04hrbe508,education,2026-03-04 11:02:48.140046
4,https://openalex.org/A5076732012,institutions.ror,https://ror.org/03cqe8w59,2026-03-03,US,Oregon State University,https://openalex.org/I131249849,https://ror.org/00ysfqy60,education,2026-03-04 11:02:48.140046
...,...,...,...,...,...,...,...,...,...,...
7705,https://openalex.org/A5049448648,institutions.ror,https://ror.org/03cqe8w59,2026-03-03,AR,Universidad Nacional de La Plata,https://openalex.org/I874386039,https://ror.org/01tjs6929,education,2026-03-04 11:02:48.140046
7706,https://openalex.org/A5049448648,institutions.ror,https://ror.org/03cqe8w59,2026-03-03,AR,Consejo Nacional de Investigaciones Científica...,https://openalex.org/I151201029,https://ror.org/03cqe8w59,funder,2026-03-04 11:02:48.140046
7707,https://openalex.org/A5063261232,institutions.ror,https://ror.org/03cqe8w59,2026-03-03,AR,Universidad Nacional de La Plata,https://openalex.org/I874386039,https://ror.org/01tjs6929,education,2026-03-04 11:02:48.140046
7708,https://openalex.org/A5054204221,institutions.ror,https://ror.org/03cqe8w59,2026-03-03,AR,Universidad Nacional de La Plata,https://openalex.org/I874386039,https://ror.org/01tjs6929,education,2026-03-04 11:02:48.140046
